# 07 — AdaBoost

**Algorithm:** Adaptive Boosting (AdaBoost)  
**Features:** TF-IDF (50K unigrams + bigrams)  
AdaBoost sequentially trains weak learners (decision stumps), re-weighting misclassified samples. The staged nature makes convergence analysis particularly informative.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, evaluate_and_print, plot_confusion_matrix,
                   plot_roc_curve, plot_pr_curve, plot_learning_curve)
from sklearn.metrics import accuracy_score
BASE_DIR, FIGURES_DIR = setup_paths()

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2, max_df=0.95)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## Model Training

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
MODEL_NAME = 'AdaBoost'
model = AdaBoostClassifier(n_estimators=100, random_state=42, algorithm='SAMME')
t0 = time.time()
model.fit(X_tr, y_train)
train_time = time.time() - t0
print(f'Training completed in {train_time:.2f}s')

In [ ]:
metrics, y_pred, y_prob = evaluate_and_print(model, X_te, y_test, MODEL_NAME, train_time)

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## Training Convergence: Staged Accuracy per Round

In [ ]:
staged_tr = [accuracy_score(y_train, p) for p in model.staged_predict(X_tr)]
staged_te = [accuracy_score(y_test,  p) for p in model.staged_predict(X_te)]
rounds = list(range(1, len(staged_tr) + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rounds, staged_tr, color='#3498db', lw=1.5, label='Train Accuracy')
ax.plot(rounds, staged_te, color='#e74c3c', lw=1.5, label='Test Accuracy')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('Accuracy')
ax.set_title('AdaBoost — Training Convergence (Staged Predictions)')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'adaboost_convergence.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak test accuracy at round {np.argmax(staged_te)+1}: {max(staged_te):.4f}')

## Estimator Weights (Importance of Each Weak Learner)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(model.estimator_weights_, color='#9b59b6', lw=1.5)
axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('Estimator Weight (alpha)')
axes[0].set_title('Estimator Weights per Round')
axes[0].grid(alpha=0.3)

axes[1].plot(model.estimator_errors_, color='#e67e22', lw=1.5)
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('Weighted Error')
axes[1].set_title('Estimator Error per Round')
axes[1].grid(alpha=0.3)

plt.suptitle('AdaBoost — Estimator Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'adaboost_estimator_diagnostics.png'), dpi=150, bbox_inches='tight')
plt.show()

## Learning Curve

In [ ]:
plot_learning_curve(model, X_tr, y_train, X_te, y_test, MODEL_NAME, FIGURES_DIR)